# Retroviral Wall Challenge — Solution

**CLS 0.7088** (nested LOFO CV)

Blend of 3 models:
1. **ElasticNet** (α=1.0, l1=0.3) on 35 tabular features (biophysical + zero-shot ESM2)
2. **Ridge** (α=12.5) on PCA3 of ESM2 **layer 12** mid-region (per-fold PCA)
3. **Ridge** (α=7.5) on PCA3 of ESM2 **layer 33** mid-region (per-fold PCA)

Key improvements over v5 baseline (CLS 0.6936):
- **Per-fold PCA**: fit PCA only on training set of each fold (no transductive leak)
- **Asymmetric Ridge regularization**: L12 α=12.5, L33 α=7.5
- Blend weights optimized by nested LOFO (inner LOFO per outer fold, step 0.05)

In [ ]:
import os, sys, warnings
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
from itertools import product as iprod
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import average_precision_score
from transformers import EsmTokenizer, EsmModel, EsmForMaskedLM

from src.metrics import compute_cls
from src.bootstrap import bootstrap_cls, print_bootstrap_results

## 1. Load data + feature engineering

In [ ]:
DATA_DIR = '../data/raw'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
splits_df = pd.read_csv(f'{DATA_DIR}/family_splits.csv')
splits = {}
for _, row in splits_df.iterrows():
    splits[row['family']] = row['rt_names'].split('|')

gt_order = pd.read_csv(f'{DATA_DIR}/rt_sequences.csv', usecols=['rt_name'])['rt_name'].tolist()
sequences = dict(zip(train['rt_name'], train['sequence']))
families = list(splits.keys())

# Missing flags
for col in ['connection_mean_pot', 'triad_best_rmsd', 'D1_D2_dist', 'D2_D3_dist',
            'yxdd_hydrophobic_fraction', 'yxdd_mean_hydrophobicity', 'yxdd_5A_mean_pot']:
    train[f'{col}_missing'] = train[col].isna().astype(int)

# Interactions
train['foldseek_gap_MMLV'] = train['foldseek_best_TM'] - train['foldseek_TM_MMLV']
train['t40_x_foldseek_MMLV'] = train['t40_raw'] * train['foldseek_TM_MMLV']
train['triad_quality'] = train['triad_found_bin'] * (1 / (train['triad_best_rmsd'].fillna(99) + 1))
train['seq_struct_compat'] = -train['perplexity'] * train['instability_index']

print(f'{len(train)} RTs, {len(families)} families')

## 2. ESM2 features (zero-shot + layer 12/33 mid-region embeddings)

In [ ]:
MODEL_NAME = 'facebook/esm2_t33_650M_UR50D'

print('Loading ESM2-650M...')
tokenizer = EsmTokenizer.from_pretrained(MODEL_NAME)
model = EsmModel.from_pretrained(MODEL_NAME)
model.eval()

# Extract mid-region embeddings for layers 12 and 33 (raw 1280D, PCA done per-fold later)
l12_raw, l33_raw = {}, {}
for rt_name, seq in sequences.items():
    inputs = tokenizer(seq, return_tensors='pt', truncation=True, max_length=1024)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        n_third = len(seq) // 3
        l12_raw[rt_name] = outputs.hidden_states[12][0, 1:-1, :][n_third:2*n_third].mean(dim=0).numpy()
        l33_raw[rt_name] = outputs.hidden_states[33][0, 1:-1, :][n_third:2*n_third].mean(dim=0).numpy()

del model
print('Layer embeddings extracted.')

# Zero-shot features (pseudo-perplexity, entropy)
model_mlm = EsmForMaskedLM.from_pretrained(MODEL_NAME)
model_mlm.eval()

for rt_name, seq in sequences.items():
    inputs = tokenizer(seq, return_tensors='pt', truncation=True, max_length=1024)
    with torch.no_grad():
        logits = model_mlm(**inputs).logits
        probs = torch.softmax(logits[0, 1:-1, :], dim=-1)
        input_ids = inputs['input_ids'][0, 1:-1]
        per_pos_ll = torch.log(probs[range(len(input_ids)), input_ids])
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
    train.loc[train['rt_name'] == rt_name, 'esm2_pseudo_ppl'] = torch.exp(-per_pos_ll.mean()).item()
    train.loc[train['rt_name'] == rt_name, 'esm2_mean_entropy'] = entropy.mean().item()
    train.loc[train['rt_name'] == rt_name, 'esm2_mean_ll'] = per_pos_ll.mean().item()

del model_mlm
print('Zero-shot features extracted.')

## 3. Feature definitions

In [ ]:
FEATURES_BASE = [c for c in [
    'foldseek_TM_MMLV', 'foldseek_TM_MMLVPE', 'foldseek_best_TM',
    'foldseek_best_LDDT', 'foldseek_best_fident', 'foldseek_TM_HIV1',
    'triad_found_bin', 'triad_best_rmsd', 'perplexity', 'log_likelihood',
    'D1_D2_dist', 'D2_D3_dist',
    't40_raw', 't45_raw', 't50_raw', 't55_raw', 't60_raw',
    'instability_index', 'gravy', 'camsol', 'net_charge',
] if c in train.columns] + [
    f'{c}_missing' for c in ['connection_mean_pot', 'triad_best_rmsd', 'D1_D2_dist',
    'D2_D3_dist', 'yxdd_hydrophobic_fraction', 'yxdd_mean_hydrophobicity', 'yxdd_5A_mean_pot']
] + ['foldseek_gap_MMLV', 't40_x_foldseek_MMLV', 'triad_quality', 'seq_struct_compat',
     'esm2_pseudo_ppl', 'esm2_mean_entropy', 'esm2_mean_ll']

# Hyperparameters (optimized via grid search over nested LOFO)
EN_ALPHA, EN_L1 = 1.0, 0.3
RIDGE_ALPHA_L12 = 12.5   # L12 needs more regularization
RIDGE_ALPHA_L33 = 7.5    # L33 needs less

def normalize(a):
    mn, mx = a.min(), a.max()
    return (a - mn) / (mx - mn) if mx - mn > 1e-12 else np.zeros_like(a)

print(f'Tabular features: {len(FEATURES_BASE)}')

## 4. Nested LOFO with per-fold PCA

In [ ]:
all_oof = []

for outer_family in families:
    outer_rts = splits[outer_family]
    outer_mask = train['rt_name'].isin(outer_rts)
    inner_mask = ~outer_mask
    inner_splits = {f: rts for f, rts in splits.items() if f != outer_family}
    inner_df = train[inner_mask].reset_index(drop=True)

    # Per-fold PCA: fit only on inner (training) set
    inner_l12 = np.array([l12_raw[n] for n in inner_df['rt_name']])
    inner_l33 = np.array([l33_raw[n] for n in inner_df['rt_name']])
    pca_l12 = PCA(n_components=3).fit(inner_l12)
    pca_l33 = PCA(n_components=3).fit(inner_l33)
    inner_l12_pca = pca_l12.transform(inner_l12)
    inner_l33_pca = pca_l33.transform(inner_l33)

    outer_l12 = np.array([l12_raw[n] for n in train.loc[outer_mask, 'rt_name']])
    outer_l33 = np.array([l33_raw[n] for n in train.loc[outer_mask, 'rt_name']])
    outer_l12_pca = pca_l12.transform(outer_l12)
    outer_l33_pca = pca_l33.transform(outer_l33)

    # Inner LOFO: generate OOF for each model
    inner_oof = {'EN': np.full(len(inner_df), np.nan),
                 'L12': np.full(len(inner_df), np.nan),
                 'L33': np.full(len(inner_df), np.nan)}

    for ifam, irts in inner_splits.items():
        tm = inner_df['rt_name'].isin(irts)
        trm = ~tm
        y = inner_df.loc[trm, 'pe_efficiency_pct'].values

        inner_oof['EN'][tm.values] = make_pipeline(
            SimpleImputer(strategy='median'), StandardScaler(),
            ElasticNet(alpha=EN_ALPHA, l1_ratio=EN_L1, max_iter=10000)
        ).fit(inner_df.loc[trm, FEATURES_BASE].values, y).predict(inner_df.loc[tm, FEATURES_BASE].values)

        inner_oof['L12'][tm.values] = make_pipeline(
            StandardScaler(), Ridge(alpha=RIDGE_ALPHA_L12)
        ).fit(inner_l12_pca[trm.values], y).predict(inner_l12_pca[tm.values])

        inner_oof['L33'][tm.values] = make_pipeline(
            StandardScaler(), Ridge(alpha=RIDGE_ALPHA_L33)
        ).fit(inner_l33_pca[trm.values], y).predict(inner_l33_pca[tm.values])

    # Find best weights on inner OOF
    best_cls, best_w = -1, (0.33, 0.33, 0.34)
    for weights in iprod(np.arange(0, 1.05, 0.05), repeat=3):
        if abs(sum(weights) - 1.0) > 0.025:
            continue
        b = weights[0]*normalize(inner_oof['EN']) + weights[1]*normalize(inner_oof['L12']) + weights[2]*normalize(inner_oof['L33'])
        y_true = inner_df['active'].values
        pe_eff = inner_df['pe_efficiency_pct'].values
        pr_auc = average_precision_score(y_true, b)
        pred_ranks = np.argsort(np.argsort(b)).astype(float)
        true_ranks = np.argsort(np.argsort(pe_eff)).astype(float)
        w_arr = (pe_eff + 0.01); w_arr = w_arr / w_arr.sum()
        mu_p, mu_t = w_arr @ pred_ranks, w_arr @ true_ranks
        dp, dt = pred_ranks - mu_p, true_ranks - mu_t
        cov = (w_arr * dp * dt).sum()
        std_p, std_t = np.sqrt((w_arr * dp**2).sum()), np.sqrt((w_arr * dt**2).sum())
        w_sp = max(cov / (std_p * std_t) if std_p > 1e-12 and std_t > 1e-12 else 0.0, 0.0)
        cls = 2 * pr_auc * w_sp / (pr_auc + w_sp) if pr_auc > 0 and w_sp > 0 else 0.0
        if cls > best_cls:
            best_cls = cls
            best_w = weights

    # Predict outer fold
    y_inner = train.loc[inner_mask, 'pe_efficiency_pct'].values

    en_pred = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
        ElasticNet(alpha=EN_ALPHA, l1_ratio=EN_L1, max_iter=10000)
    ).fit(train.loc[inner_mask, FEATURES_BASE].values, y_inner).predict(train.loc[outer_mask, FEATURES_BASE].values)

    l12_pred = make_pipeline(StandardScaler(), Ridge(alpha=RIDGE_ALPHA_L12)
    ).fit(inner_l12_pca, y_inner).predict(outer_l12_pca)

    l33_pred = make_pipeline(StandardScaler(), Ridge(alpha=RIDGE_ALPHA_L33)
    ).fit(inner_l33_pca, y_inner).predict(outer_l33_pca)

    # Normalize outer predictions using inner OOF statistics
    en_n = (en_pred - inner_oof['EN'].min()) / max(inner_oof['EN'].max() - inner_oof['EN'].min(), 1e-12)
    l12_n = (l12_pred - inner_oof['L12'].min()) / max(inner_oof['L12'].max() - inner_oof['L12'].min(), 1e-12)
    l33_n = (l33_pred - inner_oof['L33'].min()) / max(inner_oof['L33'].max() - inner_oof['L33'].min(), 1e-12)

    blended = best_w[0]*en_n + best_w[1]*l12_n + best_w[2]*l33_n

    fold_df = train.loc[outer_mask, ['rt_name', 'active', 'pe_efficiency_pct', 'rt_family']].copy()
    fold_df['predicted_score'] = blended
    all_oof.append(fold_df)

    print(f'  {outer_family:<25s} weights: EN={best_w[0]:.2f} L12={best_w[1]:.2f} L33={best_w[2]:.2f}  inner_cls={best_cls:.4f}')

oof = pd.concat(all_oof).set_index('rt_name').loc[gt_order].reset_index()
print(f'\nAll 57 OOF predictions generated.')

## 5. Score, bootstrap CI, save submission

In [ ]:
# Compute CLS
metrics = compute_cls(oof['active'].values, oof['predicted_score'].values,
                      oof['pe_efficiency_pct'].values)
print(f'PR-AUC:            {metrics["pr_auc"]:.4f}')
print(f'Weighted Spearman: {metrics["w_spearman"]:.4f}')
print(f'CLS:               {metrics["cls"]:.4f}')
print()

# Per-family PR-AUC
print('Per-family PR-AUC:')
for fam in sorted(oof['rt_family'].unique()):
    mask = oof['rt_family'] == fam
    n = mask.sum()
    na = int(oof.loc[mask, 'active'].sum())
    if 0 < na < n:
        prauc = average_precision_score(oof.loc[mask, 'active'], oof.loc[mask, 'predicted_score'])
        print(f'  {fam:<25s} n={n:2d}  active={na:2d}  PR-AUC={prauc:.3f}')
    else:
        print(f'  {fam:<25s} n={n:2d}  active={na:2d}  PR-AUC=N/A')
print()

# Bootstrap confidence intervals
print('Computing bootstrap confidence intervals...')
boot = bootstrap_cls(oof, n_bootstrap=10000)
print_bootstrap_results(boot)

# Save submission
submission = oof[['rt_name', 'predicted_score']].copy()
submission.to_csv('submission.csv', index=False)
print(f'Submission saved: {len(submission)} rows')

In [ ]:
# Pick overall best result (3-model configs + 4-model layer exploration)
all_results = {**results}
for l, res in layer_results.items():
    all_results[f'4model_L{l}'] = res

best_name = max(all_results, key=lambda k: all_results[k]['cls'])
best = all_results[best_name]
oof = best['oof']

print(f'Best configuration: {best_name}')
print(f'CLS = {best["cls"]:.4f}')
print()

# Detailed CLS breakdown
from src.metrics import compute_cls
metrics = compute_cls(oof['active'].values, oof['predicted_score'].values,
                      oof['pe_efficiency_pct'].values)
print(f'PR-AUC:            {metrics["pr_auc"]:.4f}')
print(f'Weighted Spearman: {metrics["w_spearman"]:.4f}')
print(f'CLS:               {metrics["cls"]:.4f}')
print()

# Per-family PR-AUC
from sklearn.metrics import average_precision_score
print('Per-family PR-AUC:')
for fam in sorted(oof['rt_family'].unique()):
    mask = oof['rt_family'] == fam
    n = mask.sum()
    na = int(oof.loc[mask, 'active'].sum())
    if 0 < na < n:
        prauc = average_precision_score(oof.loc[mask, 'active'], oof.loc[mask, 'predicted_score'])
        print(f'  {fam:<25s} n={n:2d}  active={na:2d}  PR-AUC={prauc:.3f}')
    else:
        print(f'  {fam:<25s} n={n:2d}  active={na:2d}  PR-AUC=N/A')
print()

# Bootstrap confidence intervals
print('Computing bootstrap confidence intervals...')
boot = bootstrap_cls(oof, n_bootstrap=10000)
print_bootstrap_results(boot)

# Save submission
submission = oof[['rt_name', 'predicted_score']].copy()
submission.to_csv('submission.csv', index=False)
print(f'Submission saved: {len(submission)} rows')

# Also save Kaggle format
kaggle = submission.copy()
kaggle.to_csv('submission_kaggle.csv', index=False)
print(f'Kaggle submission saved.')